# Lab | Data Aggregation and Filtering

In this challenge, we will continue to work with customer data from an insurance company. We will use the dataset called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by first performing data cleaning, formatting, and structuring.

In [9]:
import pandas as pd

url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv"
marketing = pd.read_csv(url)

marketing = marketing.rename(columns={
    "Customer Lifetime Value": "customer_lifetime_value",
    "Monthly Premium Auto": "monthly_premium",
    "Total Claim Amount": "total_claim_amount",
    "Policy Type": "policy_type",
    "Gender": "gender",
    "Response": "response",
    "State": "state",
    "Education": "education",
    "Effective To Date": "effective_to_date",
    "Number of Policies": "number_of_policies",
    "Sales Channel": "sales_channel"
})

marketing["effective_to_date"] = pd.to_datetime(
    marketing["effective_to_date"],
    format="%m/%d/%y"
)

marketing["month"] = marketing["effective_to_date"].dt.month_name()

1. Create a new DataFrame that only includes customers who:
   - have a **low total_claim_amount** (e.g., below $1,000),
   - have a response "Yes" to the last marketing campaign.

In [10]:
low_claims_yes = marketing[
    (marketing["total_claim_amount"] < 1000) &
    (marketing["response"] == "Yes")
].copy()

matching_records = len(low_claims_yes)
unique_customers = low_claims_yes["Customer"].nunique()

print(f"Matching records: {matching_records}")
print(f"Unique customers: {unique_customers}")

low_claims_yes.head()

Matching records: 1399
Unique customers: 1248


,Unnamed: 0,Customer,state,customer_lifetime_value,response,Coverage,education,effective_to_date,EmploymentStatus,gender,...,number_of_policies,policy_type,Policy,Renew Offer Type,sales_channel,total_claim_amount,Vehicle Class,Vehicle Size,Vehicle Type,month
3,3,XL78013,Oregon,22332.439460,Yes,Extended,College,2011-01-11,Employed,M,...,2,Corporate Auto,Corporate L3,Offer2,Branch,484.013411,Four-Door Car,Medsize,A,January
8,8,FM55990,California,5989.773931,Yes,Premium,College,2011-01-19,Employed,M,...,1,Personal Auto,Personal L1,Offer2,Branch,739.200000,Sports Car,Medsize,NaN,January
15,15,CW49887,California,4626.801093,Yes,Basic,Master,2011-01-16,Employed,F,...,1,Special Auto,Special L1,Offer2,Branch,547.200000,SUV,Medsize,NaN,January
19,19,NJ54277,California,3746.751625,Yes,Extended,College,2011-02-26,Employed,F,...,1,Personal Auto,Personal L2,Offer2,Call Center,19.575683,Two-Door Car,Large,A,February
27,27,MQ68407,Oregon,4376.363592,Yes,Premium,Bachelor,2011-02-28,Employed,F,...,1,Personal Auto,Personal L3,Offer2,Agent,60.036683,Four-Door Car,Medsize,NaN,February


### Conclusion

The filter returned **1,399 matching records**.

Because some customers appear in more than one record, these records represent **1,248 unique customers**. These customers had a total claim amount below $1,000 and responded positively to the last marketing campaign.

This group may be attractive for further marketing because it combines positive campaign engagement with relatively low claim amounts.

2. Using the original Dataframe, analyze:
   - the average `monthly_premium` and/or customer lifetime value by `policy_type` and `gender` for customers who responded "Yes", and
   - compare these insights to `total_claim_amount` patterns, and discuss which segments appear most profitable or low-risk for the company.

In [3]:
responded_yes = marketing[marketing["response"] == "Yes"]

segment_summary = (
    responded_yes
    .groupby(["policy_type", "gender"])
    .agg(
        customers=("customer_lifetime_value", "size"),
        average_monthly_premium=("monthly_premium", "mean"),
        average_customer_lifetime_value=("customer_lifetime_value", "mean"),
        median_customer_lifetime_value=("customer_lifetime_value", "median"),
        average_total_claim_amount=("total_claim_amount", "mean")
    )
    .round(2)
    .sort_values("average_customer_lifetime_value", ascending=False)
)

segment_summary

customers  average_monthly_premium  \
policy_type    gender                                       
Personal Auto  F             540                    99.00   
Special Auto   M              32                    86.34   
Corporate Auto M             154                    92.19   
               F             169                    94.30   
Special Auto   F              35                    92.31   
Personal Auto  M             536                    91.09   

                       average_customer_lifetime_value  \
policy_type    gender                                    
Personal Auto  F                               8339.79   
Special Auto   M                               8247.09   
Corporate Auto M                               7944.47   
               F                               7712.63   
Special Auto   F                               7691.58   
Personal Auto  M                               7448.38   

                       median_customer_lifetime_value  \
policy_type    gender                                   
Personal Auto  F                              5380.90   
Special Auto   M                              5394.59   
Corporate Auto M                              6047.03   
               F                              5318.90   
Special Auto   F                              5107.16   
Personal Auto  M                              5926.73   

                       average_total_claim_amount  
policy_type    gender                              
Personal Auto  F                           452.97  
Special Auto   M                           429.53  
Corporate Auto M                           408.58  
               F                           433.74  
Special Auto   F                           453.28  
Personal Auto  M                           457.01

### Conclusion

Among customers who responded positively, the **Personal Auto – Female** segment had the highest average customer lifetime value at approximately **$8,339.79**. It also had the highest average monthly premium at approximately **$99.00**.

The **Corporate Auto – Male** segment had the lowest average total claim amount at approximately **$408.58**, which may indicate a relatively lower-risk segment.

The Personal Auto – Female segment appears commercially attractive because of its high average lifetime value and premium. However, profitability cannot be confirmed because the dataset does not include operating costs, acquisition costs, or profit margins. Total claim amount should therefore be treated as a risk indicator rather than a complete profitability measure.

3. Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

In [11]:
customers_by_state = (
    marketing
    .groupby("state")["Customer"]
    .nunique()
    .reset_index(name="customer_count")
    .sort_values("customer_count", ascending=False)
)

states_over_500 = customers_by_state[
    customers_by_state["customer_count"] > 500
].reset_index(drop=True)

states_over_500

,state,customer_count
0,California,3150
1,Oregon,2601
2,Arizona,1703
3,Nevada,882
4,Washington,798


### Conclusion

Five states have more than 500 unique customers:

- California: 3,150 customers
- Oregon: 2,601 customers
- Arizona: 1,703 customers
- Nevada: 882 customers
- Washington: 798 customers

California has the largest customer base, followed by Oregon and Arizona. These states may deserve greater attention in future marketing campaigns because they represent the largest customer populations in the dataset.

4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.

In [5]:
clv_by_education_gender = (
    marketing
    .groupby(["education", "gender"])["customer_lifetime_value"]
    .agg(["max", "min", "median"])
    .round(2)
    .sort_values("median", ascending=False)
)

clv_by_education_gender

max      min   median
education            gender                            
High School or Below M       83325.38  1940.98  6286.73
                     F       55277.45  2144.92  6039.55
College              M       61134.68  1918.12  6005.85
Master               F       51016.07  2417.78  5729.86
Bachelor             F       73225.96  1904.00  5640.51
College              F       61850.19  1898.68  5623.61
Master               M       50568.26  2272.31  5579.10
Doctor               M       32677.34  2267.60  5577.67
Bachelor             M       67907.27  1898.01  5548.03
Doctor               F       44856.11  2395.57  5332.46

### Conclusion

The highest median customer lifetime value belongs to customers with a **High School or Below education level who are male**, with a median value of approximately **$6,286.73**.

The lowest median customer lifetime value belongs to **female customers with a Doctor education level**, with a median value of approximately **$5,332.46**.

The median is the most useful statistic for comparing the typical customer because it is less affected by extreme values. The maximum and minimum values show the observed range but should not be used alone to compare customer groups.

## Bonus

5. The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.

In [12]:
month_order = [
    "January", "February", "March", "April",
    "May", "June", "July", "August",
    "September", "October", "November", "December"
]

policies_by_state_month = (
    marketing
    .groupby(["state", "month"])["number_of_policies"]
    .sum()
    .unstack(fill_value=0)
    .reindex(columns=month_order, fill_value=0)
)

policies_by_state_month

month,January,February,March,April,May,June,July,August,September,October,November,December
state,,,,,,,,,,,,
Arizona,3052,2864,0,0,0,0,0,0,0,0,0,0
California,5673,4929,0,0,0,0,0,0,0,0,0,0
Nevada,1493,1278,0,0,0,0,0,0,0,0,0,0
Oregon,4697,3969,0,0,0,0,0,0,0,0,0,0
Washington,1358,1225,0,0,0,0,0,0,0,0,0,0


### Conclusion

The pivot table shows the total number of policies by state and month. In this calculation, the `number_of_policies` values were summed, so the results represent actual policy totals rather than simply counting customer records.

The dataset contains policy records for January and February only. California has the highest policy volume in both months, followed by Oregon and Arizona.

The remaining months show zero because no records are available for those months in the dataset.

6.  Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.

*Hint:*
- *To accomplish this, you will first need to group the data by state and month, then count the number of policies sold for each group. Afterwards, you will need to sort the data by the count of policies sold in descending order.*
- *Next, you will select the top 3 states with the highest number of policies sold.*
- *Finally, you will create a new DataFrame that contains the number of policies sold by month for each of the top 3 states.*

In [13]:
policy_totals_by_state = (
    marketing
    .groupby("state")["number_of_policies"]
    .sum()
    .sort_values(ascending=False)
)

top_3_states = policy_totals_by_state.head(3).index

top_3_states_by_month = policies_by_state_month.loc[top_3_states]

top_3_states_by_month

month,January,February,March,April,May,June,July,August,September,October,November,December
state,,,,,,,,,,,,
California,5673,4929,0,0,0,0,0,0,0,0,0,0
Oregon,4697,3969,0,0,0,0,0,0,0,0,0,0
Arizona,3052,2864,0,0,0,0,0,0,0,0,0,0


### Conclusion

The three states with the highest total number of policies are:

1. California
2. Oregon
3. Arizona

Their policy totals by month are:

- California: 5,673 policies in January and 4,929 in February
- Oregon: 4,697 policies in January and 3,969 in February
- Arizona: 3,052 policies in January and 2,864 in February

California has the highest policy volume in both months. These three states could be prioritized for further marketing and policy-performance analysis.

7. The marketing team wants to analyze the effect of different marketing channels on the customer response rate.

Hint: You can use melt to unpivot the data and create a table that shows the customer response rate (those who responded "Yes") by marketing channel.

In [8]:
response_rate_by_channel = (
    marketing
    .groupby("sales_channel")
    .agg(
        customers=("response", "size"),
        positive_responses=(
            "response",
            lambda responses: (responses == "Yes").sum()
        ),
        response_rate=(
            "response",
            lambda responses: (responses == "Yes").mean() * 100
        )
    )
    .sort_values("response_rate", ascending=False)
)

response_rate_by_channel["response_rate"] = (
    response_rate_by_channel["response_rate"].round(2)
)

response_rate_by_channel

,customers,positive_responses,response_rate
sales_channel,,,
Agent,4121,742,18.01
Web,1626,177,10.89
Branch,3022,326,10.79
Call Center,2141,221,10.32


### Conclusion

The Agent channel had the highest customer response rate at **18.01%**.

The response rates for the other channels were:

- Web: 10.89%
- Branch: 10.79%
- Call Center: 10.32%

The Agent channel appears to be the most effective channel for generating positive customer responses in this dataset. However, response rate alone does not show the total business impact. The marketing team should also compare the cost of each channel, the number of customers reached, and the customer lifetime value generated.

External Resources for Data Filtering: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9